# Lab 03B: A CrewAI Research Crew — Context Handoff + Memory

**Week 3 — Agentic AI: Building Autonomous Intelligent Systems**

In Lab 03A you built reflection and memory *by hand*. Now you'll use a real framework, **CrewAI**, where **agents**, **tasks**, and **memory** are first-class building blocks. You'll build a three-agent research crew — a **researcher**, an **analyst**, and a **writer** — where each agent's output becomes the next agent's **context**. Then you'll turn on CrewAI's built-in **memory** and see how it lets the crew carry context *across runs*, not just within one.

## Introduction

This lesson answers:

- What is a CrewAI crew (Agents, Tasks, Crew, Process)?
- How does one agent hand its work off to the next?
- What is the difference between **task-local state** (context within a run) and **remembered context** (memory across runs)?
- What does a framework give you that hand-writing the loop does not?

## Learning Goals

After completing this lesson you will be able to:

- Define **Agents** (role / goal / backstory) and **Tasks** (description / expected_output).
- Chain tasks so each one's output becomes the next task's **`context`**.
- Run a sequential **Crew** and read each task's output.
- Enable CrewAI **memory** and explain task-local state vs. remembered context across runs.

This lab is on the **API-key** track and talks to Gemini through CrewAI + LiteLLM.

## What is a CrewAI crew?

CrewAI models a workflow as a small team. You define **Agents** (each a persona with a `role`, `goal`, and `backstory`), give them **Tasks** (each a `description` + `expected_output`), and assemble them into a **Crew** that runs the tasks in order (`Process.sequential`). The magic is **`context`**: you list which earlier tasks a task depends on, and CrewAI feeds those outputs in automatically — that is the agent handoff.

```
   topic
     │
     ▼
   ┌──────────────┐  research brief
   │  researcher  │──────────────┐
   └──────────────┘              │  context
                                 ▼
                          ┌──────────────┐  analysis
                          │   analyst    │──────────────┐
                          └──────────────┘              │  context
                                                        ▼
                                                 ┌──────────────┐  final report
                                                 │    writer    │────────────▶
                                                 └──────────────┘
   each task's output becomes the next task's `context`.
   turn on Crew memory and that context can also persist ACROSS runs.
```

## Use cases

A sequential crew fits any task with clear, ordered phases:

- **Research -> analysis -> writing** (this lab): gather, interpret, present.
- **Draft -> review -> revise**: a writer, a critic, an editor.
- **Plan -> build -> test**: a planner, a coder, a tester.
- **Extract -> normalize -> report**: a data pipeline of specialist steps.

If steps are *independent* rather than ordered, you would fan them out in parallel instead (Week 2); a crew is for handoffs.

## Building blocks

- **Agent** — a worker defined by `role`, `goal`, and `backstory` (always a senior/expert persona here).
- **Task** — a `description` (what to do), an `expected_output` (the shape), and the `agent` that owns it.
- **`context`** — the list of earlier tasks whose outputs feed this one (the handoff).
- **Crew + Process** — the agents + tasks + an execution order (`sequential`).
- **Memory (optional)** — CrewAI's built-in short-term / long-term / entity memory, which carries context across runs.
- **An embedder** — memory stores and retrieves by similarity, so it needs an embedding model (we point it at Gemini).

## Considerations for trustworthy crews

- **Bound the crew.** More agents and hops mean more cost and more places to go wrong; add only the steps the task needs.
- **Context is not free.** Every handoff injects the prior output into the next prompt — long chains grow the context fast.
- **Memory is powerful and sticky.** Remembered context helps continuity but can also carry stale or wrong facts forward; know what is being persisted.
- **Constrain outputs where it matters.** Use `expected_output` (and schemas) so a downstream agent receives a predictable shape.
- **Keep handoffs inspectable.** Read each `task.output` so you can see exactly what each agent contributed.

## Setup: add your Gemini API key as a Colab secret

1. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey).
2. In Colab, click the **key icon** in the left sidebar ("Secrets").
3. Add a new secret named **`GEMINI_API_KEY`** and paste your key as the value.
4. Toggle **"Notebook access"** on for that secret.

The next cell installs CrewAI (which pulls in LiteLLM, the layer that lets CrewAI call Gemini). This is a larger install — give it a minute.

In [1]:
!pip install -q crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.9/185.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

> **Heads-up on the pip output:** you may see an `ERROR: pip's dependency resolver ...` line mentioning `bigframes` and `rich`. This is **expected in Colab and safe to ignore** — CrewAI upgrades `rich`, and Colab's preinstalled `bigframes` pins an older `rich`; this lab never uses `bigframes`, and the install still succeeded. If Colab shows a **"Restart session"** prompt after installing, click it (or **Runtime -> Restart session**) and re-run from the setup cell.

In [2]:
import os

from pydantic import BaseModel, Field
from typing import List
from google.colab import userdata
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()
tracing=True
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")

# LiteLLM (under CrewAI) reads the key from this env var for gemini/* models.
os.environ["GEMINI_API_KEY"] = api_key

# The `gemini/` prefix routes to the Gemini API. NOTE: this lab uses gemini-2.5-flash, not the
# gemini-2.5-flash-lite the other labs use. A crew fires many larger, multi-step requests, and the
# lite model -- the cheapest and most heavily used -- is often overloaded (HTTP 503) for requests
# that size, while flash has the capacity to serve them reliably. num_retries adds a per-call retry
# as an extra cushion for the occasional transient blip.
llm = LLM(
    model="gemini/gemini-3.1-flash-lite",
    api_key=api_key,
    temperature=0.3,
    num_retries=5,
)

# While CrewAI/LiteLLM is quietly retrying a 503, it still logs alarming red "ERROR" lines.
# Quiet those so the notebook output stays readable; our own retry messages (plain prints) still show.
import logging
for _noisy in ("crewai.flow.runtime", "LiteLLM", "litellm", "root"):
    logging.getLogger(_noisy).setLevel(logging.CRITICAL)

# ChromaDB's Google embedder (used by CrewAI memory) still imports the legacy google.generativeai
# package and prints a deprecation notice. It works fine; silence the notice to keep output clean.
import warnings
warnings.filterwarnings("ignore", message=r".*google\.generativeai.*")

# CrewAI can upload run "traces" to its hosted dashboard. We keep it off so nothing leaves this
# notebook and CrewAI stops printing its "Tracing Preference Saved" panel. (See the note by the run
# cell -- the dashboard is genuinely useful for your own later projects; flip this to "true" to try it.)
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Quick connectivity check. Gemini can return a transient 503 ("high demand") on any call,
# so we retry a few times, and if it still will not respond we warn instead of crashing.
import time

def _health_check(prompt, tries=4, wait=8):
    for attempt in range(1, tries + 1):
        try:
            return llm.call(prompt)
        except Exception as e:
            if "503" in str(e) and attempt < tries:
                print(f"  Model overloaded (503). Waiting {wait}s, then retrying ({attempt}/{tries})...")
                time.sleep(wait)
            else:
                raise

try:
    print(_health_check("Say 'Setup complete!' and nothing else."))
except Exception as e:
    print("Connectivity check could not complete (model may be busy):", str(e)[:120])
    print("That is OK -- num_retries retries a transient 503 when you run the crew below.")

Setup complete!


## The four agents

Each agent is a senior persona. They share the same model; what makes one a researcher and another a writer is the `role` / `goal` / `backstory`.

In [3]:
TOPIC = "the impact of AI agents on software development workflows"


researcher = Agent(
    role="Senior Research Analyst",
    goal="Gather a faithful, specific research brief on the topic.",
    backstory="You dig up the key facts, real challenges, and concrete examples; you cite concepts, not hype.",
    llm=llm,
    verbose=False,
)

fact_checker = Agent(
    role="Senior Fact-Checker",
    goal="Verify benchmarks, tool names, release timelines,new techniques and industry data statistics",
    backstory="You verify the key facts, collected data, new tools etc.",
    llm=llm,
    verbose=False,
)

analyst = Agent(
    role="Senior Strategy Analyst",
    goal="Turn research into ranked implications and clear recommendations.",
    backstory="You are opinionated and evidence-driven; you separate what matters from what does not.",
    llm=llm,
    verbose=False,
)

writer = Agent(
    role="Senior Technical Writer",
    goal="Turn analysis into a polished, readable short report.",
    backstory="You write tight, engaging prose for a general audience; no jargon dumps.",
    llm=llm,
    verbose=False,
)

## The three tasks -- and the handoff via `context`

Each task's `description` follows the Role / Context / Task / Constraints / Format shape. The part that makes this a *crew* and not three separate calls is the **`context=[...]`** argument on each `Task` -- that list **is** the handoff.

When you write `context=[research_task]` on the analyst's task, CrewAI takes whatever `research_task` produced and **injects it into the analyst's prompt automatically**. You never copy the brief across by hand; you just name the upstream task. The writer names both earlier tasks, so it receives both outputs.

Reading the `context=` lines top to bottom is the wiring diagram of the crew:

```
research_task    context=[]                          -> runs first; no upstream input
                       |
                       |  its output is injected as context
                       v
analysis_task    context=[research_task]             -> sees the researcher's brief
                       |
                       |  both outputs injected
                       v
writing_task     context=[research_task,             -> sees BOTH the brief
                          analysis_task]                 and the analysis
```

So the handoff is declarative: instead of passing arguments between function calls, each task *declares which earlier tasks it depends on*, and CrewAI threads the outputs through. The topic is embedded directly in the first task, so we call `kickoff()` with no templated inputs.

In [4]:
class AnalysisOutput(BaseModel):
    executive_summary: str = Field(
        description="A concise 2-sentence summary of the main strategic findings."
    )
    key_implications: List[str] = Field(
        description="A list of the top 3 ranked implications backed by logical reasoning."
    )
    recommendations: List[str] = Field(
        description="Exactly 2 actionable recommendations for software development teams."
    )

In [5]:
research_task = Task(
    description=f"""# Context
You are the first step; the analyst and writer build on your brief.

# Task
Produce a concise research brief on the topic in the Input section.

# Constraints
- Cover key facts, main challenges, and 2-3 notable examples.
- Be specific; cite concepts, not URLs.

# Format
Short sections with clear headers.

# Input (topic)
{TOPIC}""",
    expected_output="A structured research brief with clear section headers.",
    agent=researcher,
)

fact_check_task = Task(
    description=f"""# Context
The researcher's brief is available to you as context.

# Task
Validate the researcher's collected data, evidences, visuals and provide the correct data to the analyst

# Constraints
- Include only verified benchmarks, tool names, release timelines,new techniques and industry data statistics.
- Keep both the facts who have be achieved and which are yet to achive or will not achive in 2 separate sections.

# Format
A brief format with Release timelines, verified benchmarks, tool names.
Type of Traces- eg.( Data Statistics and Documentation etc)

# Input (topic)
{TOPIC}""",
    expected_output="A structured brief with clear section bullets.",
    agent=fact_checker,
    context=[research_task]
)


analysis_task = Task(
    description="""# Context
The researcher's and fact_checker's brief is available to you as context.

# Task
Analyze the Fact checkers validated data and draw the most important implications and recommendations.
Refer to the researchers task too.

# Constraints
- Rank implications by impact; back each claim with reasoning.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
    expected_output="A ranked analysis with a summary, implications, and recommendations.",
    agent=analyst,
    context=[research_task, fact_check_task],
    output_pydantic=AnalysisOutput
)

writing_task = Task(
    description="""# Context
The researcher's, fact_checker's AND the analyst brief are available to you as context.

# Task
Write a polished, publication-ready short report that combines them for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.

# Format
A title, then 2-3 short paragraphs.""",
    expected_output="A short titled report of 200-300 words.",
    agent=writer,
    context=[research_task, fact_check_task, analysis_task],
)

## A note on 503 "model overloaded" errors

Gemini can return a transient **503 ("high demand")** when a model is busy. Two choices keep this lab robust:

1. **Model choice:** this lab uses `gemini-2.5-flash` (see setup). A crew fires many larger requests, and the cheaper `gemini-2.5-flash-lite` is frequently overloaded for that load, while `flash` has the capacity to serve it.
2. **Per-call retry:** `num_retries=5` on the `LLM` retries an individual call on a transient blip before it can fail the crew.

If Gemini has a broad capacity event, calls can still fail -- that is a server-side outage, not a bug. Wait a few minutes and run the cell again.

## Run the crew

`Process.sequential` runs the tasks in order and threads the `context` through. After the run, each task's result is on `task.output` — read them to see exactly what each agent handed off.

> **Async note (Colab):** Colab already runs an `asyncio` event loop, and this version of CrewAI refuses a *synchronous* `crew.kickoff()` from inside a running loop. So we use the async entry point **`await crew.kickoff_async()`** with top-level `await` — exactly the pattern from the async lab. (In a plain `.py` script with no running loop, `crew.kickoff()` works directly.)

> **Aside — CrewAI's tracing dashboard:** CrewAI can upload a step-by-step trace of each run (every agent, the exact prompts and responses, token counts, latency) to a hosted web dashboard for debugging. We keep it **off** in this lab (`CREWAI_TRACING_ENABLED=false` in setup) so nothing leaves your notebook and the output stays clean — everything you need is printed inline below. For your own real projects it is worth a look: set `tracing=True` on the `Crew` (it needs a free CrewAI account) to inspect runs in the UI.

In [6]:
crew = Crew(
    agents=[researcher, fact_checker, analyst, writer],
    tasks=[research_task, fact_check_task, analysis_task, writing_task],
    process=Process.sequential,
    verbose=False,
)

try:
    await crew.kickoff_async()

    print("=== RESEARCH BRIEF (researcher) ===")
    print(research_task.output.raw)
    print("\n=== FACT CHECKER (fact_checker -- read the brief via context) ===")
    print(fact_check_task.output.raw)
    print("=== ANALYSIS SUMMARY ===")
    print(analysis_task.output.pydantic.executive_summary)

    print("\n=== ANALYSIS IMPLICATIONS ===")
    for implication in analysis_task.output.pydantic.key_implications:
      print(f"- {implication}")
    print("\n=== FINAL REPORT (writer -- read both via context) ===")
    print(writing_task.output.raw)
except Exception as e:
    if "503" in str(e):
        print("Gemini is under heavy load right now (repeated 503s) and the crew could not finish.")
        print("This is a transient, server-side capacity issue -- not a bug in your code or this lab.")
        print("Wait a few minutes and re-run this cell; single calls usually recover quickly.")
    else:
        raise

=== RESEARCH BRIEF (researcher) ===
### Research Brief: AI Agents in Software Development Workflows

#### Overview
AI agents represent a shift from "copilot" models (which suggest code snippets) to "autonomous" models capable of executing multi-step tasks. Unlike traditional IDE extensions, these agents operate within a loop of planning, execution, and verification, interacting with file systems, terminals, and version control to complete end-to-end development objectives.

#### Key Facts
*   **Agentic Loops:** Modern agents utilize a "ReAct" (Reasoning + Acting) framework. They decompose high-level prompts into sub-tasks, execute them, observe the output (e.g., compiler errors or test failures), and iterate until the objective is met.
*   **Context Window Management:** Agents utilize Retrieval-Augmented Generation (RAG) to index entire codebases. This allows them to maintain awareness of cross-file dependencies, which is a significant upgrade over the localized context of standard LLM

## What just happened: context handoff

You never passed the research text to the analyst by hand. Because `analysis_task` declared `context=[research_task]`, CrewAI injected the researcher's output into the analyst's prompt automatically; the writer got both. That is the crew handoff — coordination through `context` instead of manual argument passing.

But notice: that context is **task-local**. It lives only for this one `kickoff()`. Run the crew again and it starts fresh with no memory of the last run. Enabling **memory** changes that.

## Optional: enable CrewAI memory (context that survives across runs)

Set `memory=True` on the Crew and CrewAI keeps short-term, long-term, and entity memory *across* `kickoff()` calls — so a later run can recall what earlier runs established. Because memory retrieves by similarity, it needs an **embedder**; we point it at Gemini's embedding model so the lab stays Gemini-only (the default embedder is OpenAI).

The contrast to feel:
- **Without memory** (above): every run is isolated — only task-local context within the run.
- **With memory** (below): the crew accumulates a memory store that later runs read from.

> **Heads-up:** memory + the embedder config can vary by CrewAI version and needs a live Colab run to confirm. If the embedder line errors, check the CrewAI memory docs for the provider/model names your installed version expects.

> As above, we use `await memory_crew.kickoff_async()` because Colab has a running event loop.

In [7]:
memory_crew = Crew(
    agents=[researcher, fact_checker, analyst, writer],
    tasks=[research_task, fact_check_task, analysis_task, writing_task],
    process=Process.sequential,
    memory=True,  # <-- turn on short-term / long-term / entity memory across runs
    embedder={
        # CrewAI renamed the Gemini embeddings provider: use "google-generativeai" (the Gemini API
        # path) -- "google-vertex" is the separate Vertex AI path, and the old bare "google" is gone.
        "provider": "google-generativeai",
        "config": {"api_key": api_key, "model_name": "gemini-embedding-001"},
    },
    verbose=False,
)

# Run 1 seeds the memory; Run 2 can recall it. (Same crew, run twice.)
try:
    print("--- Run 1 (seeds memory) ---")
    await memory_crew.kickoff_async()
    print(writing_task.output.raw[:300], "...")

    print("\n--- Run 2 (can recall Run 1 from memory) ---")
    await memory_crew.kickoff_async()
    print(writing_task.output.raw[:300], "...")
except Exception as e:
    if "503" in str(e):
        print("Gemini is under heavy load right now (repeated 503s); the memory demo could not finish.")
        print("This is a transient server-side issue -- wait a few minutes and re-run this cell.")
    else:
        raise

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


--- Run 1 (seeds memory) ---


### The Rise of AI Agents: From Coding to Orchestration

Software development is undergoing a fundamental shift. We are moving away from manual, line-by-line coding toward an "intent-driven" model, where developers act as architects and AI agents handle the implementation. Unlike simple autocomplete

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 ...

--- Run 2 (can recall Run 1 from memory) ---


### The Rise of Autonomous Software Engineers: Promises and Pitfalls

The landscape of software development is shifting from passive code-completion tools to autonomous AI agents. Unlike standard assistants that merely suggest snippets, these new systems—such as Devin, OpenHands, and Cursor—act as j ...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
research_task.agent = None
fact_check_task.agent = None
analysis_task.agent = None
writing_task.agent = None

# 2. Instantiate the Hierarchical Crew
hierarchical_crew = Crew(
    agents=[researcher, fact_checker, analyst, writer],
    tasks=[research_task, fact_check_task, analysis_task, writing_task],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=True,
)

try:
    print("--- Executing Hierarchical Workflow ---")
    result = await hierarchical_crew.kickoff_async()

    print("\n=== FINAL GENERATED REPORT ===")
    print(result.raw)
except Exception as e:
    if "503" in str(e):
        print("Gemini is under heavy load right now. Wait a few minutes and re-run.")
    else:
        raise

--- Executing Hierarchical Workflow ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ca41e5c7-2148-4b85-a229-b1c22f4adc31                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: # Context                                                                                                │
│  You are the first step; the analyst and writer build on your brief.                                            │
│                                                                                                                 │
│  # Task                                                                                                         │
│  Produce a concise research brief on the topic in the Input section.                                            │
│                                                                                                                 │
│  # Constraints                                                                                                  │
│  - Cover key facts, main challenges, and 2-3 notable examples.                                                  │
│  - Be specific; cite concepts, not URLs.                                                                        │
│                                                                                                                 │
│  # Format                                                                                                       │
│  Short sections with clear headers.                                                                             │
│                                                                                                                 │
│  # Input (topic)                                                                                                │
│  the impact of AI agents on software development workflows                                                      │
│  ID: 4f6179b6-65b4-4ffe-a13b-2f8504cc9d7f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: # Context                                                                                                │
│  You are the first step; the analyst and writer build on your brief.                                            │
│                                                                                                                 │
│  # Task                                                                                                         │
│  Produce a concise research brief on the topic in the Input section.                                            │
│                                                                                                                 │
│  # Constraints                                                                                                  │
│  - Cover key facts, main challenges, and 2-3 notable examples.                                                  │
│  - Be specific; cite concepts, not URLs.                                                                        │
│                                                                                                                 │
│  # Format                                                                                                       │
│  Short sections with clear headers.                                                                             │
│                                                                                                                 │
│  # Input (topic)                                                                                                │
│  the impact of AI agents on software development workflows                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'The task is to produce a concise research brief on the impact of AI agents on software      │
│  development workflows. The brief must cover key facts, main challenges, and 2-3 notable examples. It...        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### The Rise of the Autonomous Developer: Navigating the Agentic Shift                                         │
│                                                                                                                 │
│  The landscape of software engineering is undergoing a fundamental transformation. We are moving beyond simple  │
│  AI-assisted coding—where tools like Copilot act as reactive pair programmers—into the era of autonomous AI     │
│  agents. Unlike their predecessors, these agents utilize "ReAct" (Reasoning + Acting) frameworks to             │
│  proactively plan tasks, execute terminal commands, and navigate entire codebases. By leveraging                │
│  Retrieval-Augmented Generation (RAG), modern agents like Devin, OpenHands, and GitHub Copilot Workspace can    │
│  maintain global context, allowing them to handle complex, multi-step engineering objectives that once          │
│  required hours of manual effort.                                                                               │
│                                                                                                                 │
│  This evolution is shifting the developer’s primary role from "writer" to "architect and reviewer." As agents   │
│  take over the heavy lifting of syntax generation, the bottleneck in software development is migrating toward   │
│  system design and integration testing. However, this power comes with significant challenges. AI agents are    │
│  inherently non-deterministic; they can struggle with "contextual drift" in legacy codebases or introduce       │
│  subtle logic errors that pass basic tests but fail in production. Furthermore, granting agents write-access    │
│  to repositories creates critical security risks, necessitating robust sandboxing and mandatory                 │
│  human-in-the-loop checkpoints.                                                                                 │
│                                                                                                                 │
│  To thrive in this new environment, organizations must treat AI agents as team members that require oversight   │
│  rather than "set-and-forget" solutions. The path forward involves building rigorous CI/CD pipelines that       │
│  serve as automated guardrails, combined with comprehensive observability frameworks that track an agent’s      │
│  reasoning, tool usage, and human interventions. By prioritizing architectural integrity and governance,        │
│  development teams can harness the speed of autonomous agents while maintaining the security and reliability    │
│  required for modern software delivery.                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: # Context                                                                                                │
│  The researcher's, fact_checker's AND the analyst brief are available to you as context.                        │
│                                                                                                                 │
│  # Task                                                                                                         │
│  Write a polished, publication-ready short report that combines them for a general audience.                    │
│                                                                                                                 │
│  # Constraints                                                                                                  │
│  - 200-300 words; clear and engaging; no jargon dumps.                                                          │
│                                                                                                                 │
│  # Format                                                                                                       │
│  A title, then 2-3 short paragraphs.                                                                            │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL GENERATED REPORT ===
### The Rise of the Autonomous Developer: Navigating the Agentic Shift

The landscape of software engineering is undergoing a fundamental transformation. We are moving beyond simple AI-assisted coding—where tools like Copilot act as reactive pair programmers—into the era of autonomous AI agents. Unlike their predecessors, these agents utilize "ReAct" (Reasoning + Acting) frameworks to proactively plan tasks, execute terminal commands, and navigate entire codebases. By leveraging Retrieval-Augmented Generation (RAG), modern agents like Devin, OpenHands, and GitHub Copilot Workspace can maintain global context, allowing them to handle complex, multi-step engineering objectives that once required hours of manual effort.

This evolution is shifting the developer’s primary role from "writer" to "architect and reviewer." As agents take over the heavy lifting of syntax generation, the bottleneck in software development is migrating toward system design and inte

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ca41e5c7-2148-4b85-a229-b1c22f4adc31                                                                       │
│  Final Output: ### The Rise of the Autonomous Developer: Navigating the Agentic Shift                           │
│                                                                                                                 │
│  The landscape of software engineering is undergoing a fundamental transformation. We are moving beyond simple  │
│  AI-assisted coding—where tools like Copilot act as reactive pair programmers—into the era of autonomous AI     │
│  agents. Unlike their predecessors, these agents utilize "ReAct" (Reasoning + Acting) frameworks to             │
│  proactively plan tasks, execute terminal commands, and navigate entire codebases. By leveraging                │
│  Retrieval-Augmented Generation (RAG), modern agents like Devin, OpenHands, and GitHub Copilot Workspace can    │
│  maintain global context, allowing them to handle complex, multi-step engineering objectives that once          │
│  required hours of manual effort.                                                                               │
│                                                                                                                 │
│  This evolution is shifting the developer’s primary role from "writer" to "architect and reviewer." As agents   │
│  take over the heavy lifting of syntax generation, the bottleneck in software development is migrating toward   │
│  system design and integration testing. However, this power comes with significant challenges. AI agents are    │
│  inherently non-deterministic; they can struggle with "contextual drift" in legacy codebases or introduce       │
│  subtle logic errors that pass basic tests but fail in production. Furthermore, granting agents write-access    │
│  to repositories creates critical security risks, necessitating robust sandboxing and mandatory                 │
│  human-in-the-loop checkpoints.                                                                                 │
│                                                                                                                 │
│  To thrive in this new environment, organizations must treat AI agents as team members that require oversight   │
│  rather than "set-and-forget" solutions. The path forward involves building rigorous CI/CD pipelines that       │
│  serve as automated guardrails, combined with comprehensive observability frameworks that track an agent’s      │
│  reasoning, tool usage, and human interventions. By prioritizing architectural integrity and governance,        │
│  development teams can harness the speed of autonomous agents while maintaining the security and reliability    │
│  required for modern software delivery.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

## Your turn (exercises)

1. **Add a fourth agent.** Insert a "Senior Fact-Checker" task between the researcher and the analyst that flags any claim the brief can't support; feed it forward with `context`.
2. **Structured handoff.** Give `analysis_task` an `output_pydantic` model so the writer receives typed, predictable analysis instead of prose.
3. **Swap the process.** Try `Process.hierarchical` (with a manager LLM) and observe how task delegation changes.
4. **Prove memory works.** With `memory=True`, run the crew on a topic, then run it again asking it to "build on what you found last time" and check whether Run 2 references Run 1.
5. **Compare to Lab 03A.** You built memory by hand there and got it from the framework here. Which was clearer? Which would you reach for in production, and why?

      ==> Framework-managed memory is superior for production because it decouples state engineering from task execution, allowing developers to focus entirely on core business logic. Furthermore, by using structured framework memory, debugging and tracking production failures becomes much cleaner, as historical context is natively cataloged and easier to isolate during system traces.

When you're done, save a copy (**File -> Save a copy in Drive**) and submit your notebook link via Canvas.